raw doging micrograd python demo but in Ruby
https://www.youtube.com/watch?v=VMj-3S1tku0&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ

In [89]:
require 'set'

class Value
    attr_reader :data, :children, :op
    attr_accessor :label, :grad

    def initialize(data, children = [], operator = '', label: '')
        @data = data
        @grad = 0.0
        @children = children.to_set
        @op = operator
        @label = label
    end

    def inspect
        # similar to python just for the lulz
        "Value:(data=#{@data})"
    end

    def +(operrand)
        Value.new( self.data + operrand.data, [self, operrand], '+')
    end

    
    def *(operrand)
        Value.new( self.data * operrand.data, [self, operrand], '*')
    end
end

:*

In [87]:
a = Value.new(2.0, label: 'a')
b = Value.new(-3.0, label: 'b')
c = Value.new(10.0, label: 'c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value.new(-2.0, label: 'f')
l = f * d; l.label = 'L' # making it lower case as 'L' will be treated as const

"L"

In [78]:
d.children

#<Set: {Value:(data=-6.0), Value:(data=10.0)}>

In [79]:
d.children.first.children

#<Set: {Value:(data=2.0), Value:(data=-3.0)}>

In [80]:
d.op

"+"

# Graphviz
in jupyter console
```sh
gem install --user-install ruby-graphviz
```

and the man himself via Docker shell
```sh
# 1. Find the container name/ID:
docker ps

# 2. Run apt as root inside the running container:
docker exec -u 0 -it <container_name_or_id> /bin/bash

# 3. Inside that root shell:
apt-get update
apt-get install -y graphviz
exit
```

In [81]:
require "ruby-graphviz"

false

In [88]:

def trace(root)
  nodes = Set.new
  edges = Set.new

  build = lambda do |v|
    unless nodes.include?(v)
      nodes.add(v)
      v.children.each do |child|
        edges.add([child, v])
        build.call(child)
      end
    end
  end

  build.call(root)
  [nodes, edges]
end

def draw_dot_graphviz(root)
  nodes, edges = trace(root)

  g = GraphViz.new(:G, type: :digraph, rankdir: "LR")
  node_map = {}

  # Create value nodes and (optional) op nodes
  nodes.each do |n|
    value_id = "v_#{n.object_id}"
    value_label = "{ #{n.label} | data #{format("%.4f", n.data)} | grad #{format("%.4f", n.grad)}}"
    value_node = g.add_nodes(value_id, label: value_label, shape: "record")
    node_map[value_id] = value_node

    unless n.op.nil? || n.op.empty?
      op_id = "op_#{n.object_id}_#{n.op}"
      op_node = g.add_nodes(op_id, label: n.op)
      node_map[op_id] = op_node
      g.add_edges(op_node, value_node)
    end
  end

  # Connect child value -> parent op
  edges.each do |n1, n2|
    from_id = "v_#{n1.object_id}"
    to_id = n2.op.nil? || n2.op.empty? ? "v_#{n2.object_id}" : "op_#{n2.object_id}_#{n2.op}"
    g.add_edges(node_map[from_id], node_map[to_id])
  end

  g

end

:draw_dot_graphviz

In [93]:
g = draw_dot_graphviz(l)
g.output(svg: "micrograd.svg")


# Backpropagation

we start at L and calc gradient along the nodes. 
deriv of each node with respect to L
In Neural Network we are interested in **"Loss Function"** - L with respect to all these **weights** - nodes 'a - f'

refer Py notes for manual  breakdown

In [94]:
d.grad = -2.0  # value of 'f'
f.grad = 4.0  # value of 'd'
l.grad = 1.0

c.grad = -2.0
e.grad = -2.0

a.grad = 6.0
b.grad = -4.0

-4.0

# Neurons

![image](https://www.researchgate.net/publication/364814302/figure/fig5/AS:11431281092677232@1666928276027/Neural-net-Structure-with-an-Activation-Function-Source-CS231n-Stanford-2017.png)


[cs231n neuron images](https://www.google.com/search?sca_esv=5aba584b8a922c2c&udm=2&fbs=ADc_l-aN0CWEZBOHjofHoaMMDiKpaEWjvZ2Py1XXV8d8KvlI3p-ML-906rRL_m6h4jR-tdCH-vUIlZq9RzugLEcfjf51b4dfDKizXS4hTwRCZW2Tyeo_h9FK7Iw3R0k8aayAiwizhG1xI1JXZfQwT07KT0cm5LxxccrVaZNLTb_5H1XATdGr7j71tLMKK8t_5Ec039O8ZXGxZyM9qt7SU8VWruIU_kGXeQ&q=cs231n+neuron&sa=X&ved=2ahUKEwiwqIawxLmTAxVCRWcHHfCSFJYQtKgLegQIDhAB&biw=1600&bih=1658&dpr=2#sv=CAMSVhoyKhBlLV80VnZNTkt2aWdESkdNMg5fNFZ2TU5LdmlnREpHTToOQTVSNkMyaVNESUljak0gBCocCgZtb3NhaWMSEGUtXzRWdk1OS3ZpZ0RKR00YADABGAcg_J7elQswAkoIEAEYASABKAE)

---

- 'w' are weights
- 'b'  in cell is 'bias'
1. we cumulate all the input with weights and add bias
2. and we produce activations func(usually some squashing func, i.e. sigmoid of tanH func)

In [1]:
# gem install --user-install iruby
# iruby register --force
# gem install --user-install numo-narray
# gem install --user-install matplotlib
require 'matplotlib/pyplot'

LoadError: cannot load such file -- matplotlib/pyplot